# PyTorch training

***

## Prerequisites

In [1]:
# [papermill] pip install disabled; using preconfigured .v3test-venv


***

## Setup Configuration file path

In [2]:
import os
import sys

module_path = os.path.abspath(os.path.join("../.."))

if module_path not in sys.path:
    sys.path.append(module_path)

# Dataset

The data set (The Social Dilemma Tweets - Text Classification 2020) was downloaded from [Kaggle](https://www.kaggle.com/datasets/kaushiksuresh147/the-social-dilemma-tweets).
This dataset brings you the twitter responses made with the #TheSocialDilemma hashtag after watching the eye-opening documentary "The Social Dilemma" released in an OTT platform(Netflix) on September 9th, 2020.
The dataset was extracted using TwitterAPI, consisting of nearly 10,526 tweets from twitter users all over the globe!

We'd like to train a model based on the content of the text in order to determine the sentiment.

This is a multi-class classification problem:
* Negative - 0
* Neutral - 1
* Positive - 2

In [3]:
! rm -rf ./data && mkdir -p ./data
! curl https://sagemaker-sample-files.s3.amazonaws.com/datasets/tabular/tweets_dataset/TheSocialDilemma.csv -o ./data/data.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

 37 6925k   37 2601k    0     0  2204k      0  0:00:03  0:00:01  0:00:02 2204k

100 6925k  100 6925k    0     0  3266k      0  0:00:02  0:00:02 --:--:-- 3266k


# Step 1 - Import Modules

Here we’ll import some libraries and define some variables.

In [4]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

In [5]:
sagemaker_client = boto3.client("sagemaker")
s3_client = boto3.client("s3")

[07/15/26 17:32:58] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=9232441;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9232442;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Create a SageMaker Session and save the default region and the execution role in some Python variables

In [6]:
import boto3
sagemaker_session = Session(boto_session=boto3.Session(region_name="us-west-1"))
region = "us-west-1"
role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=9232447;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9232448;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


In [7]:
bucket_name = sagemaker_session.default_bucket()

## Upload the dataset in the default Amazon S3 Bucket

In order to make data available for the SageMaker Processing Job, let's copy the dataset in the default S3 Bucket

In [8]:
# Download the 
# clean the buckets first
s3_client.delete_object(Bucket=bucket_name, Key="e2e-base/data/input")

input_data = sagemaker_session.upload_data('./data/data.csv', key_prefix="e2e-base/data/input")

input_data

's3://sagemaker-us-west-1-729646638167/e2e-base/data/input/data.csv'

***

# Step 2 - Create the ModelTrainer

In [9]:
! pygmentize ./scripts/train.py

zsh:1: command not found: pygmentize


In [10]:
from sagemaker.core.helper.session_helper import load_sagemaker_config
from sagemaker.core import image_uris

[07/15/26 17:33:00] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=9232453;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9232454;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [11]:
import boto3
sagemaker_session = Session(boto_session=boto3.Session(region_name="us-west-1"))

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
configs = load_sagemaker_config()


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=9232459;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9232460;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [12]:
instance_type = "ml.c5.xlarge"  # Override the instance type if you want to get a different container version
instance_count = 1

instance_type

'ml.c5.xlarge'

In [13]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.6.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

                    INFO     Defaulting to only available Python version: py312                   ]8;id=9232467;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=9232468;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

'763104351884.dkr.ecr.us-west-1.amazonaws.com/pytorch-training:2.6.0-cpu-py312'

In [14]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)

role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type, instance_count=instance_count, keep_alive_period_in_seconds=0
)

# define Training Job Name
job_name = f"train-pytorch-batch"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={"epochs": 100, "learning_rate": 0.001, "batch_size": 100},
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    role=role,
)

[07/15/26 17:33:01] INFO     SageMaker session not provided. Using default Session.                  ]8;id=9232475;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9232476;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Role                                                          ]8;id=9232483;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=9232484;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20251201T194045' validated for training. Using                          
                             it.                                                                                   

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=9232490;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9232491;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#204\204]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=9232498;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=9232499;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-1.amazonaws.com/pytorch-training:2.6                     
                             .0-cpu-py312                                                                          

In [15]:
from sagemaker.train.configs import InputData

# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=input_data,
)

# Check input channels configured
TRAINING_INPUTS = [train_input]
TRAINING_INPUTS

[InputData(channel_name='train', data_source='s3://sagemaker-us-west-1-729646638167/e2e-base/data/input/data.csv', content_type=None)]

***

## Queue Some Training Jobs
This section and the following are intended to be used interactively so that you can explore how to use the SageMaker Python SDK to submit jobs to your Batch queues. Let's start by selecting which queue to submit to.

### Select the Queue to Use

In [16]:
from sagemaker.train.aws_batch.training_queue import TrainingQueue
# Set the queue type to use for your job submission
SMTJ_BATCH_QUEUE = "ml-c5-xlarge-queue"

# Construct the queue object using the SageMaker Python SDK
queue = TrainingQueue(SMTJ_BATCH_QUEUE)
print(f"Using queue: {queue.queue_name}")

Using queue: ml-c5-xlarge-queue


### Submit your jobs
In the next cell, we are going to submit 2 Training jobs in the queue

We are going to use the API `submit` to submit all the jobs

In [17]:
for i in range(1, 3):
    job_name_i = f"{job_name}-{i}"
    queued_job = queue.submit(model_trainer, TRAINING_INPUTS, job_name_i)
    print(f"Submitted job {job_name_i}: {queued_job}")

Submitted job train-pytorch-batch-1: <sagemaker.train.aws_batch.training_queued_job.TrainingQueuedJob object at 0x123d8c530>


Submitted job train-pytorch-batch-2: <sagemaker.train.aws_batch.training_queued_job.TrainingQueuedJob object at 0x123db6750>


## Display the Status of Running and 'In Queue' Jobs
We can use the job queue list and job queue snapshot APIs to programmaticaly view a snapshot of the jobs that the queue will run next. Keep in mind that for fair-share queues this ordering is dynamic and occassionally needs to be refreshed as new jobs are submitted to the queue or as share usage changes over time.

In [18]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

======== ml-c5-xlarge-queue Submitted and Runnable  ===================================
 


Job : train-pytorch-batch-2 is SUBMITTED
Job : train-pytorch-batch-1 is RUNNABLE



======== ml-c5-xlarge-queue Queue Head as of 2026-07-15-17-33-05 =========================
 
    ... no jobs in queue ... 

======== ml-c5-xlarge-queue Queue Tail =================================================== 


======== ml-c5-xlarge-queue Starting and Running Jobs  ===================================
 


    ... no running jobs ... 




### Submit an additional job
In the next cell, we are going to submit an additional job to the queue, by using the API `submit`

In [19]:
job_name_3 = job_name + "-3"
queued_job_3 = queue.submit(
    model_trainer, TRAINING_INPUTS, job_name_3
)

## Display the Status of Running and 'In Queue' Jobs
Now we are going to see another runnable job. Given that the last job has high priority, it will be run before the `MIDPRI` and `LOWPRI` jobs

In [20]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

======== ml-c5-xlarge-queue Submitted and Runnable  ===================================
 


Job : train-pytorch-batch-3 is SUBMITTED
Job : train-pytorch-batch-1 is RUNNABLE
Job : train-pytorch-batch-2 is RUNNABLE



======== ml-c5-xlarge-queue Queue Head as of 2026-07-15-17-33-08 =========================
 
    ... no jobs in queue ... 

======== ml-c5-xlarge-queue Queue Tail =================================================== 


======== ml-c5-xlarge-queue Starting and Running Jobs  ===================================
 


    ... no running jobs ... 




## Cancel a Job in the Queue
This next cell shows how to cancel an in queue job.

In [21]:
runnable_jobs = queue.list_jobs(status="RUNNABLE")
if runnable_jobs:
    for job in runnable_jobs:
        job_to_cancel = job
        print(f"Cancelling job: {job_to_cancel.describe().get('jobName', '')}")
        job_to_cancel.terminate()

Cancelling job: train-pytorch-batch-1


Cancelling job: train-pytorch-batch-2
